# 05 — QLoRA of Mistral 7B on Kaamelott, Plato or Shakespeare

**Chapter 3** · suggested time: 40 min · **GPU required** (a free Colab T4 is enough) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/en/chapter_3/05_qlora_mistral_colab.ipynb)

![Chapter 3 roadmap](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_3/figures/00_roadmap.png)

We put everything together: the examples from notebook 02, LoRA from notebook 03, NF4 quantization from notebook 04. The model is **Mistral-7B-Instruct-v0.3** (Apache 2.0, 7.25 billion parameters).

> **Before you start (Colab):** menu *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*. Downloading the model (14.5 GB) takes 2 to 4 min.
>
> **Mistral 7B is a 2024 model**, removed from Mistral's commercial catalog (which now recommends Ministral 8B). It remains an excellent object of study: open weights, a mature LoRA/QLoRA ecosystem, and its structure is the one from chapter 2.

**By the end of this notebook, you will have:**
1. loaded a 7-billion-parameter model **in 4 bits** on a 15 GB GPU;
2. looked at its responses **before** fine-tuning;
3. checked that the loss covers **only** the response;
4. run a **QLoRA** with TRL and read the train / validation curves;
5. compared the responses **before / after** and saved the **adapter**;
6. measured the throughput (tokens/s), the peak memory and the **cost** of a training run.

In [ ]:
# 0. Installation (Colab) — versions tested together; ~1 min
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.1.0"
    if not os.path.exists("finetune.py"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
        %cd ai-masterclass/en/chapter_3

import math, json, time, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
import matplotlib.pyplot as plt
import finetune as ft

assert torch.cuda.is_available(), "No GPU: on Colab, Runtime > Change runtime type > GPU (T4)."
GPU = torch.cuda.get_device_name(0)
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere (A100, L4…) and newer; not the T4
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {GPU} ({GPU_MEMORY:.1f} GB) | compute in {'BF16' if BF16 else 'FP16 (no BF16 on this GPU)'}")
print("PyTorch", torch.__version__)

## 1. Settings

Change **`CORPUS`** to fine-tune on another text. The other settings are reasonable starting points for a T4; each choice is justified in notebooks 02 to 04.

In [ ]:
CORPUS = "kaamelott"            # "kaamelott", "plato" or "shakespeare"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Kaamelott: 38 examples -> we take them all, several epochs.
# Plato / Shakespeare: thousands of examples -> a subsample, one epoch (≈ 15-25 min on a T4).
SETTINGS = {
    "kaamelott":   dict(n_train=None, n_val=None, epochs=3, eval_steps=5),
    "plato":       dict(n_train=800, n_val=80, epochs=1, eval_steps=10),
    "shakespeare": dict(n_train=800, n_val=80, epochs=1, eval_steps=10),
}[CORPUS]

MAX_LENGTH = 512                  # notebook 02: covers ~99% of the examples
LORA_R, LORA_ALPHA = 16, 32       # notebook 04: 42 M trainable parameters
LEARNING_RATE = 2e-4              # usual order of magnitude for LoRA / QLoRA
BATCH, ACCUMULATION = 4, 4        # effective batch 16 (notebook 04, §6)
if CORPUS == "kaamelott":
    ACCUMULATION = 2              # 38 examples: effective batch 8, otherwise too few steps
ADAPTER_DIR = f"adapters/mistral7b-{CORPUS}-qlora"
SEED = 42

## 2. The data from notebook 02

The files `data/<corpus>_{train,val}.jsonl` were produced in notebook 02 (split **by scene / dialogue**). We keep only the `prompt` and `completion` columns: this is TRL's **conversational prompt-completion** format, which computes the loss **only on the completion**.

In [ ]:
from datasets import Dataset

def load(split, n):
    examples = ft.load_jsonl(f"data/{CORPUS}_{split}.jsonl")
    random.Random(SEED).shuffle(examples)
    examples = examples[:n] if n else examples
    return examples, Dataset.from_list([{"prompt": e["prompt"], "completion": e["completion"]} for e in examples])

train_ex, train_ds = load("train", SETTINGS["n_train"])
val_ex, val_ds = load("val", SETTINGS["n_val"])
print(f"{CORPUS}: {len(train_ds)} training examples, {len(val_ds)} validation examples")
print(json.dumps(train_ds[0], ensure_ascii=False, indent=1)[:800])

## 3. Load Mistral 7B in 4 bits

`BitsAndBytesConfig` follows notebook 04 exactly: NF4, double quantization, compute in BF16 (or FP16 on a T4). Quantization happens **during loading**, layer by layer: the full BF16 model is never in GPU memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.unk_token        # Mistral has no padding token: we use <unk>

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, dtype=DTYPE, device_map={"": 0},
)
model.config.use_cache = False                    # incompatible with gradient checkpointing
print(f"loaded in {time.time() - start:.0f} s")
print(f"model footprint: {model.get_memory_footprint() / 1024**3:.2f} GB (BF16: 14.5 GB)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB out of {GPU_MEMORY:.1f}")
print(model.model.layers[0].self_attn.q_proj)

`Linear4bit`: each `q_proj` stores its 16.8 M weights in 4 bits. ≈ 4 GB in total, as predicted in notebook 04.

## 4. Before fine-tuning: what does the model answer?

We keep **fixed prompts**, taken from the **validation set** (scenes never seen during training), to compare before / after. The seed is fixed: only the difference in weights changes the responses.

In [ ]:
N_PROMPTS = 4
fixed_prompts = val_ex[:N_PROMPTS]
model.eval()
before = []
for e in fixed_prompts:
    before.append(ft.generate(model, tokenizer, e["prompt"], max_new_tokens=80))
    print("CONTEXT  :", e["prompt"][0]["content"].split("\n\n", 1)[1][-300:])
    print("EXPECTED :", e["completion"][0]["content"])
    print("MISTRAL  :", before[-1])
    print("-" * 100)

The "base" Instruct model often answers **like an assistant**: it comments on the scene, offers several lines, explains the context, or writes in standard French. It may know Kaamelott or Plato, but it has not been trained to **continue this format** in this register.

## 5. Add the LoRA adapters

Exactly the `ft.apply_lora` from notebook 03, `peft` version: we freeze everything and wrap the 7 types of linear layers in the 32 blocks.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("computed in notebook 04:", f"{ft.lora_parameters(LORA_R, lora_config.target_modules):,}")
print(model.base_model.model.model.layers[0].self_attn.q_proj)

## 6. Configure training (TRL)

`SFTTrainer` (Hugging Face's **TRL** library) runs the loop from chapter 2 — forward, loss, backward, step, evaluation — with everything we have seen:

| Argument | Value | Seen in |
|---|---|---|
| `max_length` | 512 | notebook 02, length distribution |
| `completion_only_loss` | `True` | notebook 01, `-100` mask |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | 4 × 4 = 16 | notebook 04, accumulation |
| `gradient_checkpointing` | `True` | notebook 04, activations |
| `learning_rate`, `lr_scheduler_type`, `warmup_steps` | 2e-4, cosine, 10% | larger lr for LoRA (notebook 03); gradual warmup then decay |
| `optim` | `paged_adamw_8bit` | notebook 04, paged optimizers |
| `bf16` / `fp16` | depends on the GPU | T4: FP16 |

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=f"checkpoints/{CORPUS}",
    num_train_epochs=SETTINGS["epochs"],
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,                       # a number < 1 = a fraction of the steps
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUMULATION,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=BF16, fp16=not BF16,
    logging_steps=1,
    eval_strategy="steps", eval_steps=SETTINGS["eval_steps"],
    save_strategy="no",                     # we save the adapter by hand at the end
    seed=SEED, data_seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer)

if not BF16:
    # TRL converts the adapters to BF16 for a 4-bit model; the T4 does not support BF16:
    # we keep them in FP32 (the computation runs in FP16 thanks to autocast).
    for p in trainer.model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

n_steps = math.ceil(len(train_ds) / (BATCH * ACCUMULATION)) * SETTINGS["epochs"]
print(f"{n_steps} optimization steps planned (effective batch {BATCH * ACCUMULATION})")

### Check the mask before launching

A classic mistake: training on the prompt **and** the response (the model then learns to copy instructions). We take a batch as the *trainer* builds it and decode the tokens whose label is **not** `-100`:

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
ids, labels = batch["input_ids"][0], batch["labels"][0]
print("batch shape:", tuple(batch["input_ids"].shape))
print(f"tokens in the sequence: {int(batch['attention_mask'][0].sum())}, of which {int((labels != -100).sum())} count in the loss")
print("\nWHAT COUNTS IN THE LOSS:", repr(tokenizer.decode(ids[labels != -100])))
print("\nWHAT IS MASKED (start) :", repr(tokenizer.decode(ids[(labels == -100) & (batch["attention_mask"][0] == 1)])[:300]))

Only the line of dialogue (and `</s>`) counts.

## 7. Train

The validation loss **before** training is the starting point (at the start `B = 0`: it is exactly the base model). Then we launch. On a T4: a few minutes for Kaamelott, 15 to 25 min for 800 examples of Plato or Shakespeare.

In [ ]:
torch.cuda.reset_peak_memory_stats()
eval_before = trainer.evaluate()["eval_loss"]
print(f"validation loss before fine-tuning: {eval_before:.3f} (perplexity {math.exp(eval_before):.1f})")

start = time.time()
result = trainer.train()
duration = time.time() - start
eval_after = trainer.evaluate()["eval_loss"]
print(f"\ntraining: {duration / 60:.1f} min | validation loss: {eval_before:.3f} -> {eval_after:.3f} "
      f"(perplexity {math.exp(eval_before):.1f} -> {math.exp(eval_after):.1f})")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB out of {GPU_MEMORY:.1f}")

In [ ]:
hist = trainer.state.log_history
train_steps = [h["step"] for h in hist if "loss" in h]
train_loss = [h["loss"] for h in hist if "loss" in h]
val_steps = [0] + [h["step"] for h in hist if "eval_loss" in h][1:-1]
val_loss = [eval_before] + [h["eval_loss"] for h in hist if "eval_loss" in h][1:-1]
if val_steps[-1] != trainer.state.global_step:
    val_steps.append(trainer.state.global_step); val_loss.append(eval_after)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(train_steps, train_loss, alpha=0.6, label="training (per step)")
ax.plot(val_steps, val_loss, marker="o", lw=2, label="validation")
ax.set_xlabel("optimization step"); ax.set_ylabel("loss (response tokens)"); ax.legend()
ax.set_title(f"QLoRA Mistral 7B — {CORPUS}")
plt.show()

**Reading the curve** (same checklist as in chapter 2). What we expect to see:
- the **validation** loss drops clearly in the first steps: the model learns the **format** and the **register**, which is easy for a model that can already talk;
- the **training** loss is noisy (one point = one batch of 8 or 16 examples);
- on **Kaamelott**, the training loss keeps dropping while validation stalls or goes back up in the 3rd epoch: **memorization** of 38 examples (notebook 06);
- on **Plato / Shakespeare**, each example was seen only once: train and validation stay close.

## 8. After fine-tuning: same prompts, same seed

In [ ]:
model.eval()
model.config.use_cache = True
for e, resp_before in zip(fixed_prompts, before):
    print("CONTEXT  :", e["prompt"][0]["content"].split("\n\n", 1)[1][-300:])
    print("EXPECTED :", e["completion"][0]["content"])
    print("BEFORE   :", resp_before)
    print("AFTER    :", ft.generate(model, tokenizer, e["prompt"], max_new_tokens=80))
    print("-" * 100)

What should change: **a single line**, in the `NAME: text` format, in the corpus **register**, then `</s>` (the model stops). What should not fool you: a "perfect" response can be a **quotation learned by heart** — that is the topic of notebook 06.

### An invented scene

The real test: a context that exists nowhere in the corpus.

In [ ]:
# Kaamelott: "Right. Anyone got an idea to get the Grail back before tonight?" / "I've got an idea, but it's not finished."
# Plato: "Tell me, Phaedrus, do you think a machine that talks knows what it says?" / "I don't know, Socrates; it seems to me it speaks very well."
INVENTED_SCENES = {
    "kaamelott": "ARTHUR: Bon. Quelqu’un a une idée pour récupérer le Graal avant ce soir ?\nPERCEVAL: Moi j’ai une idée, mais elle est pas finie.",
    "plato": "SOCRATE: Dis-moi, Phèdre, crois-tu qu'une machine qui parle sait ce qu'elle dit ?\nPHÈDRE: Je ne sais, Socrate ; il me semble qu'elle parle fort bien.",
    "shakespeare": "HAMLET:\nWhat is this engine that doth speak like men?\nHORATIO:\nA thing of numbers, lord, and borrowed words.",
}
prompt = [{"role": "user", "content": f"{ft.INSTRUCTIONS[CORPUS]}\n\n{INVENTED_SCENES[CORPUS]}"}]
for seed in range(3):
    print(f"[{seed}]", ft.generate(model, tokenizer, prompt, max_new_tokens=80, temperature=0.8, seed=seed))

## 9. Throughput, cost, saving

The cost of a training run is predicted from the measured **throughput**, not by guessing (notebook 02):

$$\text{GPU-hours} \approx \frac{N_\text{tokens} \times N_\text{epochs}}{\text{throughput (tokens/s)} \times 3600}, \qquad \text{cost} = \text{GPU-hours} \times \text{hourly price}$$

In [ ]:
def n_tokens(e):
    return min(MAX_LENGTH, len(tokenizer.apply_chat_template(e["prompt"] + e["completion"])["input_ids"]))

tokens_seen = sum(n_tokens(e) for e in train_ex) * SETTINGS["epochs"]
throughput = tokens_seen / result.metrics["train_runtime"]
print(f"{tokens_seen:,} training tokens in {result.metrics['train_runtime']:.0f} s -> {throughput:,.0f} tokens/s on {GPU}")

all_train = ft.load_jsonl(f"data/{CORPUS}_train.jsonl")
corpus_tokens = sum(n_tokens(e) for e in all_train[:2000]) / min(2000, len(all_train)) * len(all_train)
hours = corpus_tokens / throughput / 3600
print(f"\nOne epoch on ALL of {CORPUS} ({len(all_train):,} examples, ~{corpus_tokens:,.0f} tokens): ~{hours:.1f} GPU-hour(s) on this GPU")

# Hourly rental prices, orders of magnitude (Nov. 2026): check with the provider on the day.
HOURLY_PRICE = {"T4": 0.35, "L4": 0.70, "A10": 1.00, "A100": 1.80, "H100": 2.50}
price = next((p for name, p in HOURLY_PRICE.items() if name in GPU), None)
if price:
    print(f"rental cost: ~{hours:.1f} h x {price:.2f} $/h = ~{hours * price:.2f} $ (free on Colab, within the quotas)")

We save **only the adapter** (the A and B matrices), as in notebook 03:

In [ ]:
from pathlib import Path
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
for f in sorted(Path(ADAPTER_DIR).iterdir()):
    print(f"{f.name:32s} {f.stat().st_size / 1024**2:8.1f} MB")
print(json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))["base_model_name_or_path"])

≈ 160 MB of adapter versus 14.5 GB of model. The `adapter_config.json` file records the **base model**: the adapter alone is useless without it.

**Colab's disk is wiped** at the end of the session. To keep the adapter and reuse it in notebook 06, copy it to your Google Drive (next cell) or download it.

In [ ]:
SAVE_TO_DRIVE = False          # set to True to copy the adapter to Google Drive

if IN_COLAB and SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    destination = f"/content/drive/MyDrive/ai-masterclass/{ADAPTER_DIR}"
    shutil.copytree(ADAPTER_DIR, destination, dirs_exist_ok=True)
    print("copied to", destination)
# Other option: download an archive
# import shutil; from google.colab import files
# files.download(shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR))

## Summary

- QLoRA = `BitsAndBytesConfig` (frozen NF4 base) + `LoraConfig` (adapters) + `SFTTrainer` (the loop from chapter 2). Mistral 7B fits in ~4 GB and trains on a T4.
- **Check the mask** before launching: only the response tokens count.
- The validation loss drops fast: the model first learns the **format** and the **register**.
- Cost is predicted from the **measured throughput** (tokens/s): run a small job, measure, extrapolate.
- Save the **adapter** (~160 MB), not the model.

**Common failures**

| Symptom | Likely causes | What to do |
|---|---|---|
| `CUDA out of memory` | sequences too long, batch too large | reduce `MAX_LENGTH`, then `BATCH` (and increase `ACCUMULATION`), check the 4-bit loading |
| loss `nan` | unstable FP16, lr too large | lower `LEARNING_RATE`, BF16 if the GPU allows it |
| the loss falls toward 0, the responses are quotations | overfitting, train/val leak | fewer epochs, check the split by scene (notebook 02) |
| the loss drops but the model "talks like before" | the mask is wrong, or too few steps | go back to §6, increase the steps |
| the model never stops | `</s>` missing from the target | check the chat template (notebook 01, §4) |

**Experiments** (one variable at a time, same seed, same split): `LORA_R = 8` versus `32`; `target_modules` limited to attention; 1 versus 3 epochs on Kaamelott; `CORPUS = "plato"` with 200 versus 800 examples.